In [ ]:
# Cell 1: Install dependencies
!pip install -q scanpy anndata igraph leidenalg scikit-learn scipy requests
!pip install -q cellxgene-census

In [ ]:
# Cell 2: Mount Drive and upload project files
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR    = '/content/drive/MyDrive/CellJEPA_results/transfer/'
JEPA_CKPT_DIR  = '/content/drive/MyDrive/CellJEPA_results/kidney_pretrain/'
SIGREG_CKPT_DIR = '/content/drive/MyDrive/CellJEPA_results/kidney_sigreg/'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

# Upload all .py files to /content/ before running cells below:
# cell_jepa.py, cell_sigreg.py, losses.py, preprocessing.py,
# trainer.py, metrics.py, compare_pbmc3k.py, run_ablation.py,
# pretrain_kidney.py, pretrain_kidney_sigreg.py, run_transfer.py,
# fix_kidney_gene_names.py
print('Drive mounted.')
print('Kidney Cell-JEPA checkpoints expected in:', JEPA_CKPT_DIR)
print('Kidney SIGReg checkpoints expected in:   ', SIGREG_CKPT_DIR)
print('Results will be saved to:', RESULTS_DIR)

In [ ]:
# Cell 3: Pre-train Cell-JEPA on kidney (~67 min on A100)
# SKIP if you already have kidney_pretrain_final.pt from a previous run.
import subprocess, time, threading, os

REPO_DIR = '/content'

t0 = time.time()
proc = subprocess.Popen(
    ['python3', '-u', os.path.join(REPO_DIR, 'pretrain_kidney.py'),
     '--device', 'cuda',
     '--n_epochs', '4',
     '--batch_size', '32',
     '--drive_dir', JEPA_CKPT_DIR],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1
)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t_out = threading.Thread(target=stream, args=(proc.stdout,))
t_err = threading.Thread(target=stream, args=(proc.stderr,))
t_out.start(); t_err.start()
t_out.join(); t_err.join()

rc = proc.wait()
if rc != 0:
    print(f'\n*** PROCESS EXITED WITH CODE {rc} ***')
else:
    print(f'\nDone in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 4: Pre-train SIGReg on kidney (~80 min on A100)
# Uses batch_size=32 and gradient checkpointing (3 encoder passes/step).
# SKIP if you already have kidney_sigreg_final.pt from a previous run.
import subprocess, time, threading, os

REPO_DIR = '/content'

t0 = time.time()
proc = subprocess.Popen(
    ['python3', '-u', os.path.join(REPO_DIR, 'pretrain_kidney_sigreg.py'),
     '--device', 'cuda',
     '--n_epochs', '4',
     '--batch_size', '32',
     '--drive_dir', SIGREG_CKPT_DIR],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1
)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t_out = threading.Thread(target=stream, args=(proc.stdout,))
t_err = threading.Thread(target=stream, args=(proc.stderr,))
t_out.start(); t_err.start()
t_out.join(); t_err.join()

rc = proc.wait()
if rc != 0:
    print(f'\n*** PROCESS EXITED WITH CODE {rc} ***')
else:
    print(f'\nDone in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 5: Fix gene name JSONs — convert Ensembl IDs → gene symbols
# CELLxGENE Census var_names are Ensembl IDs; PBMC-3K uses gene symbols.
# This re-fetches 500 kidney cells (~1 min) to recover the HVG symbol names
# and overwrites both gene name JSONs. The model weights are NOT changed.
# Run this even if Cells 3/4 were skipped (checkpoints already existed).
import subprocess, os

REPO_DIR = '/content'
JEPA_GENES  = os.path.join(JEPA_CKPT_DIR,   'kidney_gene_names.json')
SIG_GENES   = os.path.join(SIGREG_CKPT_DIR, 'kidney_sigreg_gene_names.json')

result = subprocess.run(
    ['python3', '-u', os.path.join(REPO_DIR, 'fix_kidney_gene_names.py'),
     '--jepa_genes',   JEPA_GENES,
     '--sigreg_genes', SIG_GENES,
     '--n_cells', '500'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-2000:])

# Verify: print first 5 gene names from each file
import json
for label, path in [('Cell-JEPA genes', JEPA_GENES), ('SIGReg genes', SIG_GENES)]:
    if os.path.exists(path):
        genes = json.load(open(path))
        print(f'{label}: {genes[:5]} ... ({len(genes)} total)')
    else:
        print(f'{label}: NOT FOUND at {path}')

In [ ]:
# Cell 6: Smoke test — scratch conditions only, 200 cells, 1 epoch (~5 min on CPU)
import subprocess, os

REPO_DIR = '/content'

result = subprocess.run(
    ['python3', '-u', os.path.join(REPO_DIR, 'run_transfer.py'),
     '--smoke_test',
     '--device', 'cpu',
     '--skip_jepa_kidney',
     '--skip_sigreg_kidney',
     '--results_file', 'results_transfer_smoke.txt'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# Cell 7: Full transfer experiment — all 4 conditions (~120 min on A100)
# Requires: Cells 3, 4, and 5 completed.
import subprocess, time, threading, os

REPO_DIR = '/content'
JEPA_CKPT  = os.path.join(JEPA_CKPT_DIR,   'kidney_pretrain_final.pt')
JEPA_GENES = os.path.join(JEPA_CKPT_DIR,   'kidney_gene_names.json')
SIG_CKPT   = os.path.join(SIGREG_CKPT_DIR, 'kidney_sigreg_final.pt')
SIG_GENES  = os.path.join(SIGREG_CKPT_DIR, 'kidney_sigreg_gene_names.json')

# Verify checkpoints and gene files exist
import json
all_ok = True
for p in [JEPA_CKPT, JEPA_GENES, SIG_CKPT, SIG_GENES]:
    exists = os.path.exists(p)
    print(f'  {"\u2713" if exists else "\u2717 MISSING"}  {p}')
    if not exists:
        all_ok = False

# Sanity-check that gene names are symbols (not Ensembl IDs)
for label, path in [('JEPA', JEPA_GENES), ('SIGReg', SIG_GENES)]:
    if os.path.exists(path):
        sample = json.load(open(path))[:3]
        looks_ensembl = all(g.startswith('ENSG') for g in sample)
        status = '\u26a0 STILL ENSEMBL IDs — re-run Cell 5' if looks_ensembl else '\u2713 gene symbols OK'
        print(f'  {label} gene names: {sample}  {status}')
        if looks_ensembl:
            all_ok = False

if not all_ok:
    print('\nFix missing files / Ensembl IDs before proceeding.')
else:
    t0 = time.time()
    proc = subprocess.Popen(
        ['python3', '-u', os.path.join(REPO_DIR, 'run_transfer.py'),
         '--device', 'cuda',
         '--pretrain_epochs', '4',
         '--finetune_epochs', '30',
         '--jepa_checkpoint',   JEPA_CKPT,
         '--jepa_genes',        JEPA_GENES,
         '--sigreg_checkpoint', SIG_CKPT,
         '--sigreg_genes',      SIG_GENES,
         '--results_file', 'results_transfer.txt'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1
    )

    def stream(pipe):
        for line in pipe:
            print(line, end='', flush=True)

    t_out = threading.Thread(target=stream, args=(proc.stdout,))
    t_err = threading.Thread(target=stream, args=(proc.stderr,))
    t_out.start(); t_err.start()
    t_out.join(); t_err.join()

    rc = proc.wait()
    if rc != 0:
        print(f'\n*** PROCESS EXITED WITH CODE {rc} — see stderr above ***')
    else:
        print(f'\nDone in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 8: Display results and plot
import os, re
import numpy as np
import matplotlib.pyplot as plt

for fname in ['results_transfer_smoke.txt', 'results_transfer.txt']:
    if os.path.exists(fname):
        print(f'--- {fname} ---')
        print(open(fname).read())

def parse_transfer_results(path):
    if not os.path.exists(path):
        return {}
    text = open(path).read()
    sections = re.split(r'Zero-shot|Fine-tuned', text)
    data = {}
    phase_keys = ['zero_shot', 'fine_tuned']
    for i, phase in enumerate(phase_keys):
        if i + 1 >= len(sections):
            break
        for line in sections[i + 1].splitlines():
            nums = re.findall(r'\d+\.\d{4}', line)
            if len(nums) == 4:
                name = line[:32].strip()
                if name and not name.startswith(('-', '=', 'M')):
                    if name not in data:
                        data[name] = {}
                    data[name][phase] = {
                        'nmi': float(nums[0]), 'ari': float(nums[1]),
                        'asw': float(nums[2]), 'avg_bio': float(nums[3])
                    }
    return data

data = parse_transfer_results('results_transfer.txt')
if data:
    models = list(data.keys())
    metrics = ['nmi', 'ari', 'asw', 'avg_bio']
    metric_labels = ['NMI', 'ARI', 'ASW', 'AvgBIO']
    phases = ['zero_shot', 'fine_tuned']
    phase_labels = ['Zero-shot', 'Fine-tuned']
    colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)
    bw = 0.6

    for ax, phase, plabel in zip(axes, phases, phase_labels):
        x = np.arange(len(models))
        for mi, (met, mlabel, col) in enumerate(zip(metrics, metric_labels, colors)):
            vals = [data[m].get(phase, {}).get(met, 0.0) for m in models]
            offset = (mi - 1.5) * bw / 4
            bars = ax.bar(x + offset, vals, bw / 4, label=mlabel, color=col, alpha=0.85)
            for bar in bars:
                h = bar.get_height()
                ax.text(bar.get_x() + bw/8, h + 0.005, f'{h:.3f}',
                        ha='center', va='bottom', fontsize=7, rotation=90)
        ax.set_xticks(x)
        ax.set_xticklabels(models, fontsize=8, rotation=15, ha='right')
        ax.set_title(f'{plabel} \u2014 Kidney \u2192 PBMC-3K Transfer')
        ax.set_ylabel('Score')
        ax.set_ylim(0, 1.15)
        ax.legend(fontsize=8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.yaxis.grid(True, linestyle='--', alpha=0.4)
        ax.set_axisbelow(True)

    fig.suptitle('Cross-Tissue Transfer: Kidney Pre-training \u2192 PBMC-3K Evaluation',
                 fontsize=11)
    plt.tight_layout()
    plt.savefig('transfer_results.png', dpi=150)
    plt.show()
    print('Saved transfer_results.png')
else:
    print('No results to plot — did Cell 7 complete?')

In [ ]:
# Cell 9: Save all results to Drive
import shutil, os

RESULTS_DIR = '/content/drive/MyDrive/CellJEPA_results/transfer/'
for f in ['results_transfer.txt', 'results_transfer_smoke.txt', 'transfer_results.png']:
    if os.path.exists(f):
        shutil.copy(f, RESULTS_DIR)
        print(f'Copied {f}')
    else:
        print(f'Not found: {f} (skipping)')
print(f'Done. Files in {RESULTS_DIR}')